# Quick Check, Live: Is This Image Related to This Caption?

**Companion notebook for the Asansol Engineering College talk — "The Mathematics Behind Visual Language Models"**

This is the live version of Slide 19. Everything here is built from the math you just saw:

- **Slide 4** — cosine similarity as "meaning closeness"
- **Slide 5** — matrices as transformations (`y = Wx + b`)
- **Slide 7** — attention as `softmax(Q x K^T) x V`

Run each cell top to bottom. No GPU needed — everything runs on CPU in a few seconds.


## Part 1 — The simplest check: cosine similarity

Ask the room first: *"If I show you an image and a caption, what's the simplest linear algebra check that they're related?"*

Below, three toy vectors stand in for an **image embedding** and two **caption embeddings** — one matching, one not. In a real VLM these vectors come from a trained vision encoder and text encoder (Slide 8), but the check on them is the same dot product / cosine similarity from Slide 4.


In [ ]:
import numpy as np

# Pretend these came out of a vision encoder + text encoder (Slide 8)
image_vec        = np.array([0.90, 0.10, 0.05])   # e.g. a photo of a dog
caption_match    = np.array([0.85, 0.15, 0.02])   # "a dog sitting on grass"
caption_mismatch = np.array([0.05, 0.10, 0.95])   # "an airplane taking off"

def cosine_similarity(a, b):
    return a @ b / (np.linalg.norm(a) * np.linalg.norm(b))

print("image vs matching caption:   ", round(cosine_similarity(image_vec, caption_match), 3))
print("image vs mismatched caption: ", round(cosine_similarity(image_vec, caption_mismatch), 3))


**Expected output:** the matching caption should score close to `1.0`, the mismatched one much lower. That single number — the cosine similarity — is the entire "are these related?" check that CLIP-style pretraining (Slide 8) is trained to make work at scale.

Try changing the numbers in `caption_match` and `caption_mismatch` and re-run the cell. Watch the similarity score move.


## Part 2 — Mini walkthrough: attention between words and image patches

Now the board exercise from Slide 19, made concrete:

- A **3-word sentence**: `["a", "red", "car"]`
- A **2-patch image**: `patch_1` (say, top-half of a photo), `patch_2` (bottom-half)
- Each of these 5 tokens gets turned into a vector (Slide 4)
- This is **cross-attention** (Slide 7): **Q** comes from the language tokens, **K** and **V** come from the image patch tokens — a word asks a question of the image, not of itself
- One attention step tells us: **which patch does each word look at most?**


In [ ]:
import numpy as np
np.random.seed(0)

# 5 tokens total: 3 words + 2 image patches, each an 8-dim toy embedding
tokens = ["a", "red", "car", "patch_1", "patch_2"]
d = 8  # embedding dimension

X = np.random.randn(len(tokens), d)

# Give "red" and "car" (a red car) embeddings close to patch_1
# (say, patch_1 is the top half of a photo showing a red car),
# and make patch_2 clearly different, so the attention result
# below is easy to sanity-check.
X[3] = np.random.randn(d) * 2.0                      # patch_1
X[4] = -X[3] + np.random.randn(d) * 0.3              # patch_2: pushed the other way
X[1] = X[3] + np.random.randn(d) * 0.2               # "red"  ~ patch_1
X[2] = X[3] + np.random.randn(d) * 0.3               # "car"  ~ patch_1
X[0] = np.random.randn(d) * 0.5                       # "a": no strong preference

print("Token order:", tokens)
print("Embedding matrix X shape:", X.shape)


In [ ]:
# Slide 5 idea: linear projections y = Wx  ->  here Q, K, V are three separate W's
# Kept close to identity so the similarity structure we built into X survives
# the projection (a real trained model's W would do this job through learning).
W_q = np.eye(d) + np.random.randn(d, d) * 0.05
W_k = np.eye(d) + np.random.randn(d, d) * 0.05
W_v = np.eye(d) + np.random.randn(d, d) * 0.05

# Cross-attention (Slide 7): Q comes from the WORD tokens only,
# K and V come from the IMAGE PATCH tokens only.
# A word is going to ask a question of the image, not of itself.
word_idx = [tokens.index(t) for t in ["a", "red", "car"]]
patch_idx = [tokens.index(t) for t in ["patch_1", "patch_2"]]

X_words = X[word_idx]    # (3, d)
X_patches = X[patch_idx] # (2, d)

Q = X_words @ W_q     # (3, d) -- one query per word
K = X_patches @ W_k   # (2, d) -- one key per patch
V = X_patches @ W_v   # (2, d) -- one value per patch

print("Q, K, V shapes:", Q.shape, K.shape, V.shape)


In [ ]:
# Slide 7: cross-attention = softmax(Q K^T / sqrt(d)) V
# Q is words-only, K/V is patches-only, so attn_weights comes out as (3 words, 2 patches) directly --
# no slicing needed, because that's what cross-attention actually is.
def attention(Q, K, V):
    scores = Q @ K.T / np.sqrt(K.shape[-1])
    weights = np.exp(scores) / np.exp(scores).sum(axis=-1, keepdims=True)
    return weights, weights @ V

attn_weights, attended = attention(Q, K, V)

# Each row is a probability distribution over patches (softmax output sums to 1) --
# not just "weights", but the actual probability the word assigns to each patch.
print("Row sums (should all be 1.0):", attn_weights.sum(axis=-1).round(6))
print()

# Which patch does each WORD look at most?
words = ["a", "red", "car"]
for i, word in enumerate(words):
    p1, p2 = attn_weights[i]
    winner = "patch_1" if p1 > p2 else "patch_2"
    print(f'"{word}":  patch_1={p1:.3f}   patch_2={p2:.3f}   -> attends most to {winner}')


**What to expect:** "red" and "car" should attend clearly more to `patch_1`, since we deliberately built their embeddings to resemble it. "a" has no such signal built in, so its split between the two patches is close to arbitrary — run the cell again with a different `np.random.seed(...)` and watch "a" flip while "red" and "car" stay locked onto `patch_1`.

This tiny 5-token example is doing exactly what happens inside a real VLM's cross-attention layer (Slide 7) — just at a scale of thousands of tokens instead of five, and with weights learned from data instead of hand-picked.


## Part 3 — See the attention weights as a heatmap

A picture is worth a thousand numbers. This plots the attention matrix from Part 2 so you can see at a glance which tokens attend to which.


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(4, 3.5))
im = ax.imshow(attn_weights, cmap="Blues")
ax.set_xticks(range(len(patch_idx)))
ax.set_yticks(range(len(words)))
ax.set_xticklabels(["patch_1", "patch_2"])
ax.set_yticklabels(words)
ax.set_xlabel("image patch (K, V)")
ax.set_ylabel("word (Q)")
ax.set_title("Cross-attention weights (Slide 7, made visible)")
plt.colorbar(im, label="attention weight")
plt.tight_layout()
plt.show()


## Bonus (optional, needs internet) — the real thing with CLIP

Everything above used toy vectors so the numbers are easy to reason about live. If you have internet access in this Colab session, this cell downloads a small pretrained **CLIP** model (the same contrastive alignment idea from Slide 8) and computes a *real* image-caption similarity score, the same way slide 8's "CLIP-style contrastive pretraining" slide described it.

This cell is optional and can be skipped if you're offline or short on time.


In [ ]:
# Optional — uncomment and run if you have internet access
# !pip install -q transformers pillow requests

# import torch
# from transformers import CLIPModel, CLIPProcessor
# from PIL import Image
# import requests

# model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
# processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

# url = "http://images.cocodataset.org/val2017/000000039769.jpg"  # two cats on a couch
# image = Image.open(requests.get(url, stream=True).raw)

# captions = ["two cats sleeping on a couch", "an airplane flying in the sky"]
# inputs = processor(text=captions, images=image, return_tensors="pt", padding=True)

# with torch.no_grad():
#     outputs = model(**inputs)
#     image_embeds = outputs.image_embeds / outputs.image_embeds.norm(dim=-1, keepdim=True)
#     text_embeds = outputs.text_embeds / outputs.text_embeds.norm(dim=-1, keepdim=True)
#     similarity = image_embeds @ text_embeds.T

# for caption, score in zip(captions, similarity[0]):
#     print(f'"{caption}":  cosine similarity = {score.item():.3f}')


## Recap

| Slide | Idea | What you just ran |
|---|---|---|
| 4 | Cosine similarity = "meaning closeness" | Part 1 |
| 5 | `y = Wx + b`, matrices as transformations | Q, K, V projections in Part 2 |
| 7 | Attention = `softmax(Q K^T) V` | Part 2's `attention()` function |
| 8 | CLIP-style contrastive pretraining | Bonus cell (real model) |

Same math, five slides earlier — now it's running.
